In [ ]:
import pandas as pd
from typing import List, Optional
from pathlib import Path
import logging
import os

os.path("./data_light_output/dpe03existant_paris_light.csv")
print('os.path("./data_light_output/dpe03existant_paris_light.csv"): ', os.path("./data_light_output/dpe03existant_paris_light.csv"))


In [ ]:
import pandas as pd
from typing import List, Optional
from pathlib import Path
import logging
import os


class Data_loader:

    COLONNES_CRITIQUES = [
        "numero_dpe",
        "numero_rpls_logement",
        "numero_immatriculation_copropriete",
        "adresse_brut",
        "code_postal_brut",
        "date_etablissement_dpe",
        "date_fin_validite_dpe",
        "annee_construction",
        "etiquette_dpe",
        "etiquette_ges",
        "surface_habitable_logement",
        "conso_5_usages_par_m2_ep",
        "cout_total_5_usages",
        "qualite_isolation_murs",
        "qualite_isolation_menuiseries",
        "type_ventilation",
        "type_energie_principale_chauffage",
        "besoin_chauffage",
        "besoin_ecs",
        "type_batiment",
        "nombre_appartement",
    ]

    def __init__(
        self,
        output_dir="./data_light_output",
        columns_df_importent: Optional[List[str]] = None,
    ):
        os.makedirs(output_dir, exist_ok=True)
        self.output_dir = output_dir
        print("self.output_dir: ", self.output_dir)
        self.columns_df_importent = columns_df_importent
        self.df_light: pd.DataFrame = None

    def create_light_df_source(self, file_name: str = None) -> str:
        file_path = None
        if file_name is not None:
            file_path = self._get_file_path(file_name)
            if not file_path:
                print("file_name Se trouve pas dans le répertoire /data_source_inputs")
                return file_path

        self.df_light = self._load_clean_csv(file_path)
        output_path = f"{self.output_dir}/{file_name}_light.csv"
        if os.path.exists(output_path):
            print(f" Le fichier {fi} existe")
            return output_path

        self.df_light.to_csv(output_path, index=False)
        print("output_path: ", output_path)
        return output_path

    # privet method
    def _get_file_path(self, file_name: str) -> Path:
        os.makedirs("./data_source_inputs", exist_ok=True)

        file_path = f"./data_source_inputs/{file_name}.csv"
        check_file_path = os.path.isfile(file_path)
        if check_file_path:
            return file_path
        return None

    def _load_clean_csv(self, file_path: str) -> pd.DataFrame:

        try:
            cols_data_load = (
                self.columns_df_importent
                if self.columns_df_importent is not None
                else self.COLONNES_CRITIQUES
            )
            df_brut = pd.read_csv(file_path, usecols=cols_data_load, low_memory=False)
            df_brut = df_brut.drop_duplicates(subset=["numero_dpe"])
            df_brut = df_brut.dropna(subset=["etiquette_dpe"])
            df_brut["code_postal_brut"] = df_brut["code_postal_brut"].astype("category")
            df_brut["numero_dpe"] = df_brut["numero_dpe"].astype("category")
            df_brut["etiquette_dpe"] = df_brut["etiquette_dpe"].astype("category")
            df_brut["etiquette_ges"] = df_brut["etiquette_ges"].astype("category")

            return df_brut

        except Exception as e:
            raise e
        else:
            print("pas de data ")


if __name__ == "__main__":
    d1 = Data_loader("./data_light_output")
    pathName = d1.create_light_df_source("dpe03existant_paris")
    print("pathName: ", pathName)

self.output_dir:  ./data_light_output
 Le fichier dpe03existant_paris existe
pathName:  ./data_light_output/dpe03existant_paris_light.csv


In [ ]:
import pandas as pd
from typing import List, Optional
from pathlib import Path
import logging
import os


class Data_recontracter:

    def __init__(
        self,
        data_light_source_path="./data_light_output/dpe03existant_paris_light.csv",
        columns_df_importent: Optional[List[str]] = None,
    ):

        self.columns_df_importent = columns_df_importent
        self.data_light_source_path: pd.DataFrame = data_light_source_path
        self.df_light: pd.DataFrame = None

    def create_light_df_source(self, file_name: str = None) -> str:
        file_path = None
        if file_name is not None:
            file_path = self._get_file_path(file_name)
            if not file_path:
                print("file_name Se trouve pas dans le répertoire /data_source_inputs")
                return file_path

        self.df_light = self._load_clean_csv(file_path)
        output_path = f"{self.output_dir}/{file_name}_light.csv"
        if os.path.exists(output_path):
            print(f" Le fichier {fi} existe")
            return output_path

        self.df_light.to_csv(output_path, index=False)
        print("output_path: ", output_path)
        return output_path

    # privet method
    def _add_computed_columns(self) -> Path:
        new_df = None

        self.df_light["statut_juridique"] = "Sociale"

        condition_sociale = self.df_light["numero_rpls_logement"].notna()
        condition_privet = self.df_light["numero_immatriculation_copropriete"].notna()

        self.df_light.loc[condition_sociale, "statut_juridique"] = "Sociale"
        self.df_light.loc[condition_privet, "statut_juridique"] = "Privet"

        is_renovated = self.df_light["etiquette_dpe"].isin(["A", "B", "C", "D"])
        self.df_light["is_renovated"] = is_renovated.astype(int)
        
        print("is_renovated: ", is_renovated)
        # cp_group
        renovation_stats_by_cp_group = self.df_light.groupby("code_postal_brut")

        renovation_stats_by_cp_group["is_renovated"].agg(
            {"moyenne": "mean", "total": "sum"}
        )

        renovation_stats_by_cp_group["pourcentage_renovated"] = (
            renovation_stats_by_cp_group["moyenne"] * 100
        )

        renovation_stats_by_cp_group["pourcentage_renovated"].round(1)
        print("renovation_stats_by_cp_group: ", renovation_stats_by_cp_group)
        
        
        #cp_status_group
        renovation_stats_by_cp_status_group = self.df_light.groupby(
            ["code_postal_brut", "statut_juridique"]
        )

        renovation_stats_series = renovation_stats_by_cp_status_group[
            "is_renovated"
        ].agg({"moyenne": "mean"})

        renovation_stats_series["pourcentage_renovated"] = (
            renovation_stats_series["moyenne"] * 100
        ).round(1)




= 
df['code_postal_brut'].map(
    renovation_pivot.get_level_values(1).map({
        'Social': renovation_pivot.loc[:, ('mean', 'Social')] * 100
    })
    if ('mean', 'Social') in renovation_pivot.columns else 0
)


        os.makedirs("./data_source_inputs", exist_ok=True)

        file_path = f"./data_source_inputs/{file_name}.csv"
        check_file_path = os.path.isfile(file_path)
        if check_file_path:
            return file_path
        return None

    def _load_light_csv(self) -> pd.DataFrame:

        try:
            cols_data_load = (
                self.columns_df_importent
                if self.columns_df_importent is not None
                else self.COLONNES_CRITIQUES
            )
            self.df_light = pd.read_csv(self.data_light_source_path)
            self._add_computed_columns()

            return df_brut

        except Exception as e:
            raise e
        else:
            print("pas de data ")


if __name__ == "__main__":
    d1 = Data_loader("./data_light_output")
    pathName = d1.create_light_df_source("dpe03existant_paris")
    print("pathName: ", pathName)

In [22]:
import pandas as pd
import numpy as np

df_base = pd.DataFrame({
    'ID': [1, 2, 3, 4, 5],
    'Nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'Age': [25, 30, 22, 35, 28],
    # 'Ville': ['Paris', 'Lyon', 'Paris', 'Marseille', 'Lyon'],
    # 'Salaire': [50000, 60000, 45000, 70000, np.nan] # np.nan pour une valeur manquante
})
# print("DataFrame de base :")
print(df_base)
# df_base[['Nom','Age']]
# df_base['Seniority'] = [3, 8, 2, 10, 5]
# df_base['Age'].mean()
df_base.loc[:,('Age','Nom')]

   ID      Nom  Age
0   1    Alice   25
1   2      Bob   30
2   3  Charlie   22
3   4    David   35
4   5      Eve   28


,Age,Nom
0,25,Alice
1,30,Bob
2,22,Charlie
3,35,David
4,28,Eve


In [21]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Adresse': ['1 rue de la Paix', '2 av. des Champs', '3 bd. Voltaire'],
    'code_postal_brut': [75013, 75020, 75013],
    'Surface': [100, 150, 80]
})
print("1. DataFrame 'df' initial :")
print(df)
print("-" * 30)

1. DataFrame 'df' initial :
            Adresse  code_postal_brut  Surface
0  1 rue de la Paix             75013      100
1  2 av. des Champs             75020      150
2    3 bd. Voltaire             75013       80
------------------------------


In [77]:
import pandas as pd
import numpy as np

df_base = pd.DataFrame({
    'ID': [1, 2, 3, 4, 5],
    'Nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'Age': [25, 30, 22, 35, 28],
    'Ville': ['Paris', 'Lyon', 'Paris', 'Marseille', 'Lyon'],
    'Statut_Logement': ['Social', 'Privé', 'Social', 'Privé', 'Social'], # Nouveau : type de logement
    'Is_Renovated': [1, 0, 1, 1, 0] # Nouveau : 1 si rénové, 0 sinon
})
# Remplace les noms de colonnes pour qu'ils ressemblent plus à ton cas :
df_base.rename(columns={'Ville': 'Code_Postal_Brut', 'Statut_Logement': 'Statut_Juridique'}, inplace=True)

# print("DataFrame de base (version enrichie) :")
print(df_base)
renovation_grouped = df_base.groupby(['Code_Postal_Brut', 'Statut_Juridique'])
# print('renovation_grouped: ', renovation_grouped)
res=renovation_grouped['Age'].mean()

print( res.unstack())
res.unstack().columns
pd.MultiIndex.from_product([['mean'], ('mean',  'Privé')])
# res.unstack().columns=pd.MultiIndex.from_product([['mean'], res.unstack().columns])
index=res.unstack().columns=pd.MultiIndex.from_product([['mean'], ('mean',  'Privé')])
index
print('index: ', index)
columns=res.unstack().columns
columns=index
print('columns: ', columns)
index_columns_sociale=columns.get_level_values(0)
index_columns_privet=columns.get_level_values(-1)
print('index_columns_sociale: ', index_columns_sociale)
print('index_columns_sociale: ', index_columns_privet)


   ID      Nom  Age Code_Postal_Brut Statut_Juridique  Is_Renovated
0   1    Alice   25            Paris           Social             1
1   2      Bob   30             Lyon            Privé             0
2   3  Charlie   22            Paris           Social             1
3   4    David   35        Marseille            Privé             1
4   5      Eve   28             Lyon           Social             0
Statut_Juridique  Privé  Social
Code_Postal_Brut               
Lyon               30.0    28.0
Marseille          35.0     NaN
Paris               NaN    23.5
index:  MultiIndex([('mean',  'mean'),
            ('mean', 'Privé')],
           )
columns:  MultiIndex([('mean',  'mean'),
            ('mean', 'Privé')],
           )
index_columns_sociale:  Index(['mean', 'mean'], dtype='object')
index_columns_sociale:  Index(['mean', 'Privé'], dtype='object')


In [ ]:
import pandas as pd
from typing import List, Optional
from pathlib import Path
import logging
import os


class Data_recontracter:

    def __init__(
        self,
        data_light_source_path="./data_light_output/dpe03existant_paris_light.csv",
        columns_df_importent: Optional[List[str]] = None,
    ):

        self.columns_df_importent = columns_df_importent
        self.data_light_source_path: pd.DataFrame = data_light_source_path
        self.df_light: pd.DataFrame = None

    def create_light_df_source(self, file_name: str = None) -> str:
        file_path = None
        if file_name is not None:
            file_path = self._get_file_path(file_name)
            if not file_path:
                print("file_name Se trouve pas dans le répertoire /data_source_inputs")
                return file_path

        self.df_light = self._load_clean_csv(file_path)
        output_path = f"{self.output_dir}/{file_name}_light.csv"
        if os.path.exists(output_path):
            print(f" Le fichier {fi} existe")
            return output_path

        self.df_light.to_csv(output_path, index=False)
        print("output_path: ", output_path)
        return output_path

    # privet method
    def _add_computed_columns(self) -> Path:
        new_df = None

        self.df_light["statut_juridique"] = "Sociale"

        condition_sociale = self.df_light["numero_rpls_logement"].notna()
        condition_privet = self.df_light["numero_immatriculation_copropriete"].notna()

        self.df_light.loc[condition_sociale, "statut_juridique"] = "Sociale"
        self.df_light.loc[condition_privet, "statut_juridique"] = "Privet"

        is_renovated = self.df_light["etiquette_dpe"].isin(["A", "B", "C", "D"])
        self.df_light["is_renovated"] = is_renovated.astype(int)
        
        print("is_renovated: ", is_renovated)
        # cp_group
        renovation_stats_by_cp_group = self.df_light.groupby("code_postal_brut")

        renovation_stats_by_cp_group["is_renovated"].agg(
            {"moyenne": "mean", "total": "sum"}
        )

        renovation_stats_by_cp_group["pourcentage_renovated"] = (
            renovation_stats_by_cp_group["moyenne"] * 100
        )

        renovation_stats_by_cp_group["pourcentage_renovated"].round(1)
        print("renovation_stats_by_cp_group: ", renovation_stats_by_cp_group)
        
        
        #cp_status_group
        renovation_stats_by_cp_status_group = self.df_light.groupby(
            ["code_postal_brut", "statut_juridique"]
        )

        renovation_stats_series = renovation_stats_by_cp_status_group[
            "is_renovated"
        ].agg({"moyenne": "mean"})

        renovation_stats_series["pourcentage_renovated"] = (
            renovation_stats_series["moyenne"] * 100
        ).round(1)





df['code_postal_brut'].map(
    renovation_pivot.get_level_values(1).map({
        'Social': renovation_pivot.loc[:, ('mean', 'Social')] * 100
    })
    if ('mean', 'Social') in renovation_pivot.columns else 0
)


        os.makedirs("./data_source_inputs", exist_ok=True)

        file_path = f"./data_source_inputs/{file_name}.csv"
        check_file_path = os.path.isfile(file_path)
        if check_file_path:
            return file_path
        return None

    def _load_light_csv(self) -> pd.DataFrame:

        try:
            cols_data_load = (
                self.columns_df_importent
                if self.columns_df_importent is not None
                else self.COLONNES_CRITIQUES
            )
            self.df_light = pd.read_csv(self.data_light_source_path)
            self._add_computed_columns()

            return df_brut

        except Exception as e:
            raise e
        else:
            print("pas de data ")


if __name__ == "__main__":
    d1 = Data_loader("./data_light_output")
    pathName = d1.create_light_df_source("dpe03existant_paris")
    print("pathName: ", pathName)

In [ ]:
from data_laoder_cleaner import Data_loader
from data_make_new_df_DPE import Make_new_df
from table_factory import TableFactory
import pandas as pd
from typing import List, Optional
from pathlib import Path

import os


def start_pipeline_DPE():
   
    data_loader=Data_loader()
    data_loader.create_light_df_source(file_name='dpe03existant_paris.csv')
    Make_new_df()
    TableFactory()
    
    return 


start_pipeline_DPE()